# 任务三：比较不同优化算法的影响

这个任务的核心目标是：

在“网络结构、激活函数、训练轮数都保持不变”的前提下，
只改变优化器，然后观察不同优化器对训练效果的影响。

这次固定不变的条件有：

- 数据集：Fashion-MNIST
- 网络结构：`784 -> 256 -> 10`
- 隐藏层激活函数：ReLU
- 训练轮数：20
- batch size：64

唯一变化的是优化算法：

- SGD
- SGD with Nesterov Momentum
- Adam
- RMSprop

这个任务主要比较 3 件事：

1. 收敛速度谁更快
2. 最终验证集准确率谁更高
3. 训练过程是否稳定


## 第 1 步：导入库并设置实验环境

这一格和前两个任务差不多，因为基础框架是一样的。

需要特别关注的是 `OPTIMIZER_FACTORIES`。

这个字典把四种优化器统一管理起来，后面我们只需要循环它，
就能依次训练四组模型。

四种优化器参数分别是：

- `SGD`
  学习率 `0.01`，动量 `0.9`
- `SGD_Nesterov`
  学习率 `0.01`，动量 `0.9`，并打开 `nesterov=True`
- `Adam`
  学习率 `0.001`
- `RMSprop`
  学习率 `0.001`


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import copy
import random
import time
from pathlib import Path
from typing import Optional

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

sns.set_theme(style="whitegrid")
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

RANDOM_SEED = 42
BATCH_SIZE = 64
NUM_EPOCHS = 20
HIDDEN_DIM = 256
VALID_RATIO = 0.1

# 任务要求比较的四种优化器
OPTIMIZER_FACTORIES = {
    "SGD": lambda params: torch.optim.SGD(params, lr=0.01, momentum=0.9),
    "SGD_Nesterov": lambda params: torch.optim.SGD(
        params, lr=0.01, momentum=0.9, nesterov=True
    ),
    "Adam": lambda params: torch.optim.Adam(params, lr=0.001),
    "RMSprop": lambda params: torch.optim.RMSprop(params, lr=0.001),
}


def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 第 2 步：准备数据

这里仍然使用 Fashion-MNIST，并沿用和前两个任务相同的数据划分方式：

- 原始训练集继续拆成训练集和验证集
- 验证集比例为 10%
- 测试集保持原始测试集不变

为什么任务三还要保持完全一样的数据划分？

因为我们现在想比较的是“优化器的影响”，
如果数据划分也变了，那最后结果里就混进了别的变量，不够公平。


In [ ]:
def build_dataloaders(data_root: Path):
    transform = transforms.ToTensor()

    train_full = datasets.FashionMNIST(
        root=str(data_root),
        train=True,
        transform=transform,
        download=False,
    )
    test_dataset = datasets.FashionMNIST(
        root=str(data_root),
        train=False,
        transform=transform,
        download=False,
    )

    valid_size = int(len(train_full) * VALID_RATIO)
    train_size = len(train_full) - valid_size
    split_generator = torch.Generator().manual_seed(RANDOM_SEED)
    train_dataset, valid_dataset = random_split(
        train_full,
        [train_size, valid_size],
        generator=split_generator,
    )

    loader_kwargs = {
        "batch_size": BATCH_SIZE,
        "num_workers": 0,
        "pin_memory": torch.cuda.is_available(),
    }

    train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
    valid_loader = DataLoader(valid_dataset, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)
    return train_loader, valid_loader, test_loader


set_seed(RANDOM_SEED)
device = get_device()
data_root = Path.cwd() / "fashion_mnist"

train_loader, valid_loader, test_loader = build_dataloaders(data_root)

print("当前设备：", device)
print("数据目录：", data_root)
print("对比优化器：", list(OPTIMIZER_FACTORIES.keys()))
print(f"训练集批次数：{len(train_loader)}")
print(f"验证集批次数：{len(valid_loader)}")
print(f"测试集批次数：{len(test_loader)}")


## 第 3 步：定义固定结构的 MLP

题目要求任务三只改优化器，不改网络结构。

所以这里模型结构是固定的：

`输入层(784) -> 隐藏层(256, ReLU) -> 输出层(10)`

这和任务一的基础模型保持一致，
这样最后的性能差异就更能说明是优化器带来的，而不是模型结构变化带来的。


In [ ]:
class OptimizerMLP(nn.Module):
    def __init__(self, hidden_dim: int = HIDDEN_DIM, num_classes: int = 10) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


demo_model = OptimizerMLP()
print(demo_model)


## 第 4 步：定义训练与验证函数

这一部分和前面任务的套路一样：

- `run_one_epoch()` 负责跑一轮
- `train_model()` 负责完整训练某一个优化器对应的模型

不同的是，`train_model()` 现在会接收：

- `optimizer_name`
  当前优化器的名字，方便打印和记录
- `optimizer_factory`
  真正用来创建优化器的函数

这样写的好处是：
四种优化器都可以复用完全一样的训练流程，只是入口参数不同。


### 重点理解：什么叫“优化器工厂”

比如：

`lambda params: torch.optim.Adam(params, lr=0.001)`

这句话的意思不是马上创建 Adam，
而是先准备一个“创建 Adam 的方法”。

真正等到训练某个模型时，再把模型参数 `params` 传进去，
才真正得到一个优化器对象。

这样比提前全都创建好更方便，因为每个模型都需要自己独立的优化器。


In [ ]:
def run_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    optimizer: Optional[torch.optim.Optimizer] = None,
):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.set_grad_enabled(is_train):
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples


def train_model(
    optimizer_name: str,
    optimizer_factory,
    train_loader: DataLoader,
    valid_loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
):
    # 每一组实验开始前都重新固定随机种子，尽量让对比更公平
    set_seed(RANDOM_SEED)

    model = OptimizerMLP().to(device)
    optimizer = optimizer_factory(model.parameters())

    history = []
    best_state_dict = copy.deepcopy(model.state_dict())
    best_valid_acc = 0.0

    print(f"\n========== 开始训练：{optimizer_name} ==========")

    for epoch in range(1, NUM_EPOCHS + 1):
        start_time = time.time()

        train_loss, train_acc = run_one_epoch(
            model=model,
            dataloader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
        )
        valid_loss, valid_acc = run_one_epoch(
            model=model,
            dataloader=valid_loader,
            criterion=criterion,
            optimizer=None,
            device=device,
        )

        if valid_acc > best_valid_acc:
            best_valid_acc = valid_acc
            best_state_dict = copy.deepcopy(model.state_dict())

        epoch_seconds = time.time() - start_time
        history.append(
            {
                "optimizer": optimizer_name,
                "epoch": epoch,
                "train_loss": train_loss,
                "train_accuracy": train_acc,
                "valid_loss": valid_loss,
                "valid_accuracy": valid_acc,
                "seconds": epoch_seconds,
            }
        )

        print(
            f"{optimizer_name} | Epoch {epoch:02d}/{NUM_EPOCHS} | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
            f"valid_loss={valid_loss:.4f}, valid_acc={valid_acc:.4f} | "
            f"time={epoch_seconds:.2f}s"
        )

    model.load_state_dict(best_state_dict)
    return model, pd.DataFrame(history)


## 第 5 步：开始训练四组模型

这一格是任务三的核心。

我们会依次训练：

- SGD
- SGD with Nesterov Momentum
- Adam
- RMSprop

训练时你可以重点观察：

- 哪个优化器前几轮下降更快
- 哪个优化器验证集准确率提升更稳定
- 哪个优化器更容易出现震荡


In [ ]:
def evaluate_model(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
):
    return run_one_epoch(
        model=model,
        dataloader=dataloader,
        criterion=criterion,
        optimizer=None,
        device=device,
    )


histories = {}
test_results = {}
criterion = nn.CrossEntropyLoss()

for optimizer_name, optimizer_factory in OPTIMIZER_FACTORIES.items():
    model, history_df = train_model(
        optimizer_name=optimizer_name,
        optimizer_factory=optimizer_factory,
        train_loader=train_loader,
        valid_loader=valid_loader,
        criterion=criterion,
        device=device,
    )
    histories[optimizer_name] = history_df
    test_results[optimizer_name] = evaluate_model(
        model=model,
        dataloader=test_loader,
        criterion=criterion,
        device=device,
    )


## 第 6 步：整理对比结果表

这一格把四种优化器的关键结果整理成一张表。

重点看这些指标：

- `Best Valid Accuracy`
  验证集最高准确率
- `Epoch of Best Valid Accuracy`
  最佳验证准确率出现在第几轮
- `Test Accuracy`
  测试集最终准确率

其中最有参考价值的一般是：

- 收敛速度
- 最高验证准确率
- 曲线是否稳定


In [ ]:
def summarize_results(histories: dict[str, pd.DataFrame], test_results: dict[str, tuple[float, float]]) -> pd.DataFrame:
    summary_rows = []

    for optimizer_name, history_df in histories.items():
        best_row = history_df.loc[history_df["valid_accuracy"].idxmax()]
        test_loss, test_acc = test_results[optimizer_name]

        summary_rows.append(
            {
                "Optimizer": optimizer_name,
                "Best Valid Accuracy": best_row["valid_accuracy"],
                "Epoch of Best Valid Accuracy": int(best_row["epoch"]),
                "Final Train Loss": history_df.iloc[-1]["train_loss"],
                "Final Valid Loss": history_df.iloc[-1]["valid_loss"],
                "Final Valid Accuracy": history_df.iloc[-1]["valid_accuracy"],
                "Test Accuracy": test_acc,
                "Test Loss": test_loss,
            }
        )

    summary_df = pd.DataFrame(summary_rows).sort_values(
        "Best Valid Accuracy", ascending=False
    ).reset_index(drop=True)
    return summary_df


summary_df = summarize_results(histories, test_results)

print("优化器对比结果汇总：")
display(summary_df)


## 第 7 步：绘制对比曲线

题目让我们重点观察：

- 收敛速度与最终性能
- 训练过程的稳定性

所以这里画 3 张最有代表性的图：

- 训练损失对比
- 验证损失对比
- 验证准确率对比

后面写实验分析时，这三张图就是最直接的依据。


In [ ]:
def plot_comparison(histories: dict[str, pd.DataFrame]) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for optimizer_name, history_df in histories.items():
        axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label=optimizer_name)
        axes[1].plot(history_df["epoch"], history_df["valid_loss"], marker="o", label=optimizer_name)
        axes[2].plot(history_df["epoch"], history_df["valid_accuracy"], marker="o", label=optimizer_name)

    axes[0].set_title("不同优化器的训练损失对比")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Train Loss")
    axes[0].legend()

    axes[1].set_title("不同优化器的验证损失对比")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Valid Loss")
    axes[1].legend()

    axes[2].set_title("不同优化器的验证准确率对比")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Valid Accuracy")
    axes[2].set_ylim(0, 1.0)
    axes[2].legend()

    plt.tight_layout()
    plt.show()


plot_comparison(histories)


## 实验小结

任务三完成以后，你在写实验分析时可以重点回答这几个问题：

1. 哪个优化器收敛最快？
2. 哪个优化器验证集最高准确率更高？
3. 哪个优化器曲线更稳定？
4. SGD、Adam、RMSprop 各自表现出什么特点？

一般来说：

- SGD 往往更依赖学习率设置，前期可能慢一些
- 带 Nesterov 的 SGD 有时会比普通 SGD 更快一点
- Adam 通常收敛较快，也比较稳
- RMSprop 在某些情况下也会比较快，但最终效果要以实验结果为准

最后结论还是要根据你自己跑出来的曲线和对比表来写。
